In [1]:
import pandas as pd

In [2]:
# Read in data
df = pd.read_csv("census.csv")

In [3]:
# View overview of columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlgt           32561 non-null  int64 
 3   education       32561 non-null  object
 4   education-num   32561 non-null  int64 
 5   marital-status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital-gain    32561 non-null  int64 
 11  capital-loss    32561 non-null  int64 
 12  hours-per-week  32561 non-null  int64 
 13  native-country  32561 non-null  object
 14  salary          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


Various numeric and categorical columns with no apparent missing values.

In [4]:
# Peek at data
df.head()

,age,workclass,fnlgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [5]:
# View descriptive stats of numeric columns
df.describe()

,age,fnlgt,education-num,capital-gain,capital-loss,hours-per-week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [6]:
# Assess outliers as defined as 1.5*IQR beyond Q1 and Q3
def view_outliers_for_IQR_method(df_og, df_desc):
    """
    Provides output that lists the number of positive and negative outliers,
    the range of values of those outliers, and the total number of outliers
    for each numeric column contained in def_desc. Outliers are calculated as
    those values which are 1.5 * IQR outside of Q1 and Q3.

    df_og: pd.DataFrame of the original dataset
    df_desc: pd.DataFrame equivalent to df_og.describe()    
    """
    

    count = 0
    
    for col in df_desc.columns:
        Q1 = df_desc[col]["25%"]
        Q3 = df_desc[col]["75%"]
        IQR = Q3 - Q1
        LB = Q1 - (1.5 * IQR)
        UB = Q3 + (1.5 * IQR)
        
        outliers_pos = sum(df_og[col]>UB)
        outliers_pos_min = df_og[df_og[col]>UB][col].min()
        outliers_pos_max = df_og[df_og[col]>UB][col].max()
        outliers_neg = sum(df_og[col]<LB)
        outliers_neg_min = df_og[df_og[col]<LB][col].min()
        outliers_neg_max = df_og[df_og[col]<LB][col].max()
        
        count += outliers_pos + outliers_neg
        
        print(f"{col.upper()}")
        print(f"Total Outliers: {outliers_pos + outliers_neg}")
        print(f"Positive Outliers: {outliers_pos}")
        print(f"Range: {outliers_pos_min:.4f} - {outliers_pos_max:.4f}")
        print(f"Negative Outliers: {outliers_neg}")
        print(f"Range: {outliers_neg_min:.4f} - {outliers_neg_max:.4f}")
        print()
    
    print(f"Total Outliers: {count}")

In [7]:
dfd = df.describe()
view_outliers_for_IQR_method(df, dfd)

AGE
Total Outliers: 143
Positive Outliers: 143
Range: 79.0000 - 90.0000
Negative Outliers: 0
Range: nan - nan

FNLGT
Total Outliers: 992
Positive Outliers: 992
Range: 415913.0000 - 1484705.0000
Negative Outliers: 0
Range: nan - nan

EDUCATION-NUM
Total Outliers: 1198
Positive Outliers: 0
Range: nan - nan
Negative Outliers: 1198
Range: 1.0000 - 4.0000

CAPITAL-GAIN
Total Outliers: 2712
Positive Outliers: 2712
Range: 114.0000 - 99999.0000
Negative Outliers: 0
Range: nan - nan

CAPITAL-LOSS
Total Outliers: 1519
Positive Outliers: 1519
Range: 155.0000 - 4356.0000
Negative Outliers: 0
Range: nan - nan

HOURS-PER-WEEK
Total Outliers: 9008
Positive Outliers: 3492
Range: 53.0000 - 99.0000
Negative Outliers: 5516
Range: 1.0000 - 32.0000

Total Outliers: 15572


In [8]:
# Assess outliers as defined as +/- 2 std deviations from the mean
def view_outliers_for_std_dev_method(df_og, df_desc):
    """
    Provides output that lists the number of positive and negative outliers,
    the range of values of those outliers, and the total number of outliers
    for each numeric column contained in def_desc. Outliers are calculated as
    those values which are beyond the mean +/- (2 * std dev).

    df_og: pd.DataFrame of the original dataset
    df_desc: pd.DataFrame equivalent to df_og.describe()    
    """

    
    count = 0
    for col in df_desc.columns:
        mean = df_desc[col]["mean"]
        std = df_desc[col]["std"]
        std_2pos = mean + (2 * std)
        std_2neg = mean - (2 * std)
        
        outliers_pos = sum(df_og[col]>std_2pos)
        outliers_pos_min = df_og[df_og[col]>std_2pos][col].min()
        outliers_pos_max = df_og[df_og[col]>std_2pos][col].max()
        outliers_neg = sum(df_og[col]<std_2neg)
        outliers_neg_min = df_og[df_og[col]<std_2neg][col].min()
        outliers_neg_max = df_og[df_og[col]<std_2neg][col].max()
        
        count += outliers_pos + outliers_neg
        print(f"{col.upper()}")
        print(f"Total Outliers: {outliers_pos + outliers_neg}")
        print(f"Positive Outliers: {outliers_pos}")
        print(f"Range: {outliers_pos_min:.4f} - {outliers_pos_max:.4f}")
        print(f"Negative Outliers: {outliers_neg}")
        print(f"Range: {outliers_neg_min:.4f} - {outliers_neg_max:.4f}")
        print()
    
    print(f"Total Outliers: {count}")

In [9]:
dfd = df.describe()
view_outliers_for_std_dev_method(df, dfd)

AGE
Total Outliers: 1158
Positive Outliers: 1158
Range: 66.0000 - 90.0000
Negative Outliers: 0
Range: nan - nan

FNLGT
Total Outliers: 1249
Positive Outliers: 1249
Range: 400943.0000 - 1484705.0000
Negative Outliers: 0
Range: nan - nan

EDUCATION-NUM
Total Outliers: 1611
Positive Outliers: 413
Range: 16.0000 - 16.0000
Negative Outliers: 1198
Range: 1.0000 - 4.0000

CAPITAL-GAIN
Total Outliers: 255
Positive Outliers: 255
Range: 18481.0000 - 99999.0000
Negative Outliers: 0
Range: nan - nan

CAPITAL-LOSS
Total Outliers: 1485
Positive Outliers: 1485
Range: 974.0000 - 4356.0000
Negative Outliers: 0
Range: nan - nan

HOURS-PER-WEEK
Total Outliers: 2203
Positive Outliers: 822
Range: 66.0000 - 99.0000
Negative Outliers: 1381
Range: 1.0000 - 15.0000

Total Outliers: 7961


In [10]:
# List unique values for all columns
for col in df.columns:
    print(f"{col}: {df[col].unique()}")

age: [39 50 38 53 28 37 49 52 31 42 30 23 32 40 34 25 43 54 35 59 56 19 20 45
 22 48 21 24 57 44 41 29 18 47 46 36 79 27 67 33 76 17 55 61 70 64 71 68
 66 51 58 26 60 90 75 65 77 62 63 80 72 74 69 73 81 78 88 82 83 84 85 86
 87]
workclass: ['State-gov' 'Self-emp-not-inc' 'Private' 'Federal-gov' 'Local-gov' '?'
 'Self-emp-inc' 'Without-pay' 'Never-worked']
fnlgt: [ 77516  83311 215646 ...  34066  84661 257302]
education: ['Bachelors' 'HS-grad' '11th' 'Masters' '9th' 'Some-college' 'Assoc-acdm'
 'Assoc-voc' '7th-8th' 'Doctorate' 'Prof-school' '5th-6th' '10th'
 '1st-4th' 'Preschool' '12th']
education-num: [13  9  7 14  5 10 12 11  4 16 15  3  6  2  1  8]
marital-status: ['Never-married' 'Married-civ-spouse' 'Divorced' 'Married-spouse-absent'
 'Separated' 'Married-AF-spouse' 'Widowed']
occupation: ['Adm-clerical' 'Exec-managerial' 'Handlers-cleaners' 'Prof-specialty'
 'Other-service' 'Sales' 'Craft-repair' 'Transport-moving'
 'Farming-fishing' 'Machine-op-inspct' 'Tech-support' '?'
 'Prote

In [11]:
# Investigate class imbalance
def get_target_class_imbalance(df_og):
    low = sum(df_og["salary"] == "<=50K")
    high = sum(df_og["salary"] == ">50K")
    print(f"Samples with <=50K: {low} ({low/df_og.shape[0] * 100:.2f}%)")
    print(f"Samples with >50K: {high} ({high/df_og.shape[0] * 100:.2f}%)")

In [12]:
get_target_class_imbalance(df)

Samples with <=50K: 24720 (75.92%)
Samples with >50K: 7841 (24.08%)


In [13]:
# Assess missing values (?)
df_dropna = df.replace("?", pd.NA)

In [14]:
df_dropna.isna().sum()

age                  0
workclass         1836
fnlgt                0
education            0
education-num        0
marital-status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     583
salary               0
dtype: int64

In [15]:
df_dropna = df_dropna.dropna()
df_dropna.shape

(30162, 15)

Evaluate affect on df stats, class imbalance, and outliers

In [17]:
df_dropna.describe()

,age,fnlgt,education-num,capital-gain,capital-loss,hours-per-week
count,30162.000000,3.016200e+04,30162.000000,30162.000000,30162.000000,30162.000000
mean,38.437902,1.897938e+05,10.121312,1092.007858,88.372489,40.931238
std,13.134665,1.056530e+05,2.549995,7406.346497,404.298370,11.979984
min,17.000000,1.376900e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.176272e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.784250e+05,10.000000,0.000000,0.000000,40.000000
75%,47.000000,2.376285e+05,13.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [18]:
get_target_class_imbalance(df_dropna)

Samples with <=50K: 22654 (75.11%)
Samples with >50K: 7508 (24.89%)


In [19]:
dfd = df_dropna.describe()
view_outliers_for_IQR_method(df_dropna, dfd)

AGE
Total Outliers: 169
Positive Outliers: 169
Range: 76.0000 - 90.0000
Negative Outliers: 0
Range: nan - nan

FNLGT
Total Outliers: 903
Positive Outliers: 903
Range: 417657.0000 - 1484705.0000
Negative Outliers: 0
Range: nan - nan

EDUCATION-NUM
Total Outliers: 196
Positive Outliers: 0
Range: nan - nan
Negative Outliers: 196
Range: 1.0000 - 2.0000

CAPITAL-GAIN
Total Outliers: 2538
Positive Outliers: 2538
Range: 114.0000 - 99999.0000
Negative Outliers: 0
Range: nan - nan

CAPITAL-LOSS
Total Outliers: 1427
Positive Outliers: 1427
Range: 155.0000 - 4356.0000
Negative Outliers: 0
Range: nan - nan

HOURS-PER-WEEK
Total Outliers: 7953
Positive Outliers: 3327
Range: 53.0000 - 99.0000
Negative Outliers: 4626
Range: 1.0000 - 32.0000

Total Outliers: 13186


In [20]:
dfd = df_dropna.describe()
view_outliers_for_std_dev_method(df_dropna, dfd)

AGE
Total Outliers: 975
Positive Outliers: 975
Range: 65.0000 - 90.0000
Negative Outliers: 0
Range: nan - nan

FNLGT
Total Outliers: 1152
Positive Outliers: 1152
Range: 401104.0000 - 1484705.0000
Negative Outliers: 0
Range: nan - nan

EDUCATION-NUM
Total Outliers: 1871
Positive Outliers: 375
Range: 16.0000 - 16.0000
Negative Outliers: 1496
Range: 1.0000 - 5.0000

CAPITAL-GAIN
Total Outliers: 234
Positive Outliers: 234
Range: 18481.0000 - 99999.0000
Negative Outliers: 0
Range: nan - nan

CAPITAL-LOSS
Total Outliers: 1395
Positive Outliers: 1395
Range: 974.0000 - 4356.0000
Negative Outliers: 0
Range: nan - nan

HOURS-PER-WEEK
Total Outliers: 2239
Positive Outliers: 1009
Range: 65.0000 - 99.0000
Negative Outliers: 1230
Range: 1.0000 - 16.0000

Total Outliers: 7866


**Initial Observations**
- Missing Values
  - Encoded as `?` for `workclass`, `occupation`, and `native-country`
  - After removing missing values, class imbalances, outliers, and descriptive statistics of the new dataframe appear relatively similar to the original dataframe, suggesting that removed samples likely were not associated with a unique sub-population. 
- Encoding
  - Binary Encoding: `Salary`, `Sex`
  - Categorical columns should be one-hot encoded
    - `education` is ordinal, but will probably one-hot encode since `education-num` is also provided
- Feature Distribution
  - All numeric features are skewed right to some degree. `capital-gain` and `capital-loss` are most heavily skewed right along with `hours-per-week`.
  - `hours-per-week` had the most outliers across both calculation methods.
  - `age` had the fewest outliers according to the IQR method
  - `capital-gain` had the fewest outliers according to the 2 std dev method
  - Will proceed assuming that all outliers are valid data points to be included in analysis and modeling.
- Target variable is `salary` according to data source
- Class Imbalance
  - Will need to stratify by y (`salary`)